# Random Forest Regressor for Trip Delay Prediction

This notebook trains a baseline `RandomForestRegressor` to predict delay duration in minutes for a given trip-stop event.

The target (`delay_min`) is constructed by:
1. Merging events with GTFS stop times on `trip_id`, `stop_id`, and `stop_sequence`
2. Comparing observed event time against scheduled arrival/departure time

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
import joblib

DATA_DIR = Path("../Datasets")
EVENTS_FILE = DATA_DIR / "2022-01_HREvents.csv"
STOP_TIMES_FILE = DATA_DIR / "20220103_stop_times.txt"

# Optionally cap rows for faster iteration while developing.
# Set to None to use all rows.
MAX_ROWS = 400_000

RANDOM_STATE = 42

In [4]:
def hhmmss_to_seconds(value: str) -> float:
    """Convert HH:MM:SS (including hours >= 24) to total seconds."""
    if pd.isna(value):
        return np.nan
    parts = str(value).split(":")
    if len(parts) != 3:
        return np.nan
    h, m, s = map(int, parts)
    return h * 3600 + m * 60 + s


# Override with key-type normalization to prevent merge dtype errors.
def load_and_prepare(events_file: Path, stop_times_file: Path, max_rows=None) -> pd.DataFrame:
    events = pd.read_csv(events_file, low_memory=False)
    stop_times = pd.read_csv(stop_times_file, low_memory=False)

    stop_times = stop_times.drop(
        columns=["stop_headsign", "continuous_pickup", "continuous_drop_off"],
        errors="ignore",
    )

    merge_keys = ["trip_id", "stop_id", "stop_sequence"]

    for key in ["trip_id", "stop_id"]:
        events[key] = events[key].astype("string").str.strip()
        stop_times[key] = stop_times[key].astype("string").str.strip()

    events["stop_sequence"] = pd.to_numeric(events["stop_sequence"], errors="coerce").astype("Int64")
    stop_times["stop_sequence"] = pd.to_numeric(stop_times["stop_sequence"], errors="coerce").astype("Int64")

    events = events.dropna(subset=merge_keys)
    stop_times = stop_times.dropna(subset=merge_keys)

    stop_times["arrival_time_sec"] = stop_times["arrival_time"].apply(hhmmss_to_seconds)
    stop_times["departure_time_sec"] = stop_times["departure_time"].apply(hhmmss_to_seconds)

    merged = events.merge(
        stop_times,
        on=merge_keys,
        how="left",
    )

    merged = merged.dropna(
        subset=[
            "event_type",
            "event_time_sec",
            "arrival_time_sec",
            "departure_time_sec",
            "service_date",
        ]
    )

    merged["delay_sec"] = np.where(
        merged["event_type"] == "ARR",
        merged["event_time_sec"] - merged["arrival_time_sec"],
        merged["event_time_sec"] - merged["departure_time_sec"],
    )
    merged["delay_min"] = merged["delay_sec"] / 60.0

    if max_rows is not None and len(merged) > max_rows:
        merged = merged.sample(n=max_rows, random_state=RANDOM_STATE)

    return merged.reset_index(drop=True)

In [5]:
df = load_and_prepare(EVENTS_FILE, STOP_TIMES_FILE, max_rows=MAX_ROWS)
print(f"Prepared rows: {len(df):,}")
df[["service_date", "route_id", "trip_id", "stop_id", "stop_sequence", "event_type", "delay_min"]].head()

Prepared rows: 400,000


,service_date,route_id,trip_id,stop_id,stop_sequence,event_type,delay_min
0,2022-01-21,Blue,50444999,70049,50,DEP,0.183333
1,2022-01-20,Red,50034951,70067,30,ARR,5.900000
2,2022-01-08,Orange,49812330-HayOL,70027,130,ARR,10.216667
3,2022-01-22,Red,49812804,70091,160,DEP,0.700000
4,2022-01-02,Orange,49812638,70024,70,ARR,3.800000


In [6]:
model_df = df.copy()

# Keep arrival-only events for a cleaner and more consistent label.
model_df = model_df[model_df["event_type"] == "ARR"].copy()

model_df["service_date"] = pd.to_datetime(model_df["service_date"], errors="coerce")
model_df = model_df.dropna(subset=["service_date"])

model_df["day_of_week"] = model_df["service_date"].dt.dayofweek
model_df["month"] = model_df["service_date"].dt.month
model_df["day_of_month"] = model_df["service_date"].dt.day
model_df["is_weekend"] = (model_df["day_of_week"] >= 5).astype(int)
model_df["scheduled_hour"] = (model_df["arrival_time_sec"] // 3600).astype(int)
model_df["time_bucket_5min"] = (model_df["arrival_time_sec"] // 300).astype(int)

feature_cols = [
    "route_id",
    "trip_id",
    "stop_id",
    "stop_sequence",
    "arrival_time_sec",
    "departure_time_sec",
    "day_of_week",
    "month",
    "day_of_month",
    "is_weekend",
    "scheduled_hour",
    "time_bucket_5min",
]

target_col = "delay_min"

model_df = model_df.dropna(subset=feature_cols + [target_col]).copy()
X = model_df[feature_cols]
y = model_df[target_col]

X.head()

,route_id,trip_id,stop_id,stop_sequence,arrival_time_sec,departure_time_sec,day_of_week,month,day_of_month,is_weekend,scheduled_hour,time_bucket_5min
1,Red,50034951,70067,30,70140.0,70140.0,3,1,20,0,19,233
2,Orange,49812330-HayOL,70027,130,70920.0,70920.0,5,1,8,1,19,236
4,Orange,49812638,70024,70,67440.0,67440.0,6,1,2,1,18,224
5,Orange,49812271,70008,150,41160.0,41160.0,5,1,22,1,11,137
10,Red,50034664,70071,50,26280.0,26280.0,1,1,4,0,7,87


In [7]:
# Time-aware split by service_date: train on earlier dates, test on later dates.
ordered_dates = np.sort(model_df["service_date"].dt.date.unique())
split_index = int(len(ordered_dates) * 0.8)

train_dates = set(ordered_dates[:split_index])
test_dates = set(ordered_dates[split_index:])

train_mask = model_df["service_date"].dt.date.isin(train_dates)
test_mask = model_df["service_date"].dt.date.isin(test_dates)

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

categorical_features = ["route_id", "trip_id", "stop_id"]
numeric_features = [c for c in feature_cols if c not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
            categorical_features,
        ),
        ("num", "passthrough", numeric_features),
    ]
)

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("regressor", rf),
    ]
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Train rows: {len(X_train):,}")
print(f"Test rows:  {len(X_test):,}")
print(f"MAE:  {mae:.3f} minutes")
print(f"RMSE: {rmse:.3f} minutes")
print(f"R2:   {r2:.3f}")

Train rows: 167,021
Test rows:  32,998
MAE:  11.366 minutes
RMSE: 29.251 minutes
R2:   -0.061


In [8]:
importances = model.named_steps["regressor"].feature_importances_
importance_df = pd.DataFrame(
    {
        "feature": categorical_features + numeric_features,
        "importance": importances,
    }
).sort_values("importance", ascending=False)

importance_df.head(12)

,feature,importance
1,trip_id,0.409702
8,day_of_month,0.190797
4,arrival_time_sec,0.108268
5,departure_time_sec,0.105833
6,day_of_week,0.058213
3,stop_sequence,0.053075
11,time_bucket_5min,0.040241
2,stop_id,0.024208
10,scheduled_hour,0.007877
0,route_id,0.001555


In [ ]:
def predict_delay_minutes(
    trained_model: Pipeline,
    route_id,
    trip_id,
    stop_id,
    stop_sequence,
    arrival_time_sec,
    departure_time_sec,
    service_date,
):
    service_date = pd.to_datetime(service_date)
    row = pd.DataFrame(
        [
            {
                "route_id": route_id,
                "trip_id": trip_id,
                "stop_id": stop_id,
                "stop_sequence": stop_sequence,
                "arrival_time_sec": arrival_time_sec,
                "departure_time_sec": departure_time_sec,
                "day_of_week": service_date.dayofweek,
                "month": service_date.month,
                "day_of_month": service_date.day,
                "is_weekend": int(service_date.dayofweek >= 5),
                "scheduled_hour": int(arrival_time_sec // 3600),
                "time_bucket_5min": int(arrival_time_sec // 300),
            }
        ]
    )
    return float(trained_model.predict(row)[0])


# Example: reuse one known sample row as an input template.
example = X_test.iloc[0]
example_date = model_df.loc[example.name, "service_date"]

pred = predict_delay_minutes(
    trained_model=model,
    route_id=example["route_id"],
    trip_id=example["trip_id"],
    stop_id=example["stop_id"],
    stop_sequence=int(example["stop_sequence"]),
    arrival_time_sec=float(example["arrival_time_sec"]),
    departure_time_sec=float(example["departure_time_sec"]),
    service_date=example_date,
)

print(f"Predicted delay: {pred:.2f} minutes")
print(f"Actual delay:    {y_test.iloc[0]:.2f} minutes")

# output_path = Path("./random_forest_delay_model.joblib")
# joblib.dump(model, output_path)
# print(f"Saved model to: {output_path.resolve()}")

Predicted delay: 3.76 minutes
Actual delay:    7.28 minutes
Saved model to: /Users/thomasyoung/data/NU/2026-Spring/CS4100/final/CS4100_Final/src/random_forest_delay_model.joblib
